# A second grid: IEEE 118

Does what we found on the Texas grid hold on another grid and another GENCO model? This notebook repeats the core of
`frozen_test` and `load_response` on the IEEE 118-bus system, with the GENCO model the authors released for it
(`gridfm/genco-pf-datakit`, `case118_ieee/base/seed0`). That model was trained on single outages (N-1: one line or one
generator). Every scenario of its dataset has such an outage, so the operating points with all lines in service each
have one generator out; every contingency below adds lines to that, which takes the model beyond what it saw.

1. **Operating points the model never saw.** Only scenarios in the authors' own test split are used
   (`case118_ieee_scenario_splits.json`, published with their evaluation run). Before anything else, the notebook checks
   that our network model reproduces the stored AC solutions and that GENCO reproduces the authors' test accuracy.
2. **Contingencies, fixed before the run.** Per operating point: one, three and six random lines, three and six lines
   nearest a random bus, and a six-line corridor.
3. **The same workflow and the same definitions as on Texas.** Newton-Raphson from GENCO's state, then the classical
   starts; three outcomes (verified safe, verified violation, unresolved). GENCO alone means its prediction taken at face
   value. The reference is the classical solution.
4. **The load-growth probe.** GENCO's voltage response to a 2 % growth of every load against the AC response.

Runtime: about 60 MB of downloads and 10 to 20 minutes; a GPU is not needed. Output: `second_grid.zip`, also saved to
`MyDrive/genco_runs`.

## Settings (frozen)

In [ ]:
MODEL_REPO = "gridfm/genco-pf-datakit"
MODEL_REVISION = "f5eb20ee798b9847b723c91874af85d32a96a468"
MODEL_DIR = "case118_ieee/base/seed0"
MODEL_CONFIG = "case118_ieee_base_seed0.yaml"
EVAL_RUN = "mlflow/eval/case118_ieee/base/seed0/8289694ec85b4df080e584d18a815b57"   # the authors' evaluation of this model
DATA_REPO = "gridfm/pf_small_case118_ieee"
DATA_REVISION = "3f5a6e266f759be15e3777bf4d70133ff40e6fa7"
NETWORK = "case118_ieee"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit used for Texas
LOCAL_MODEL, LOCAL_DATA = "genco-pf-datakit", "pf_small_case118_ieee"
OUT_DIR = "second_grid"

N_PARTITIONS = 24                     # data partitions read, spread evenly over the dataset
N_BASES = 150                         # operating points, drawn from the authors' test split
FAMILIES = [("uniform", 1), ("uniform", 3), ("uniform", 6), ("nearest", 3), ("nearest", 6), ("corridor", 6)]
REGION_HOPS = 4                       # corridor: the region around the centre bus, grown until it holds 3k candidate lines
POLICY = {"tol_pu": 1e-8, "max_iter": 30, "order": ["genco", "base", "dc", "flat"], "line_limit": 1.0, "voltage_buses": "pq",
          "low_voltage_flag": 0.5, "recovery": {"max_iter": 100, "homotopy_steps": [0.25, 0.5, 0.75, 1.0]},
          "enforce_q_limits": False, "q_limit_rounds": 15}
GROWTH = 0.02                         # load-growth probe: every load +/- 2 %, as on Texas
N_GROWTH = 60                         # operating points used for the probe
AUTHORS_RMSE_VM_PQ = 0.000275         # the authors' test RMSE of Vm at PQ buses for this model (their RMSE.csv, checked below)
MODEL_CHECK_FACTOR = 5                # the model check fails if our RMSE is more than this many times theirs
NETWORK_CHECK_PU = 1e-4               # the network check fails if a stored AC solution has a larger mismatch in our model
BOOTSTRAP = 1000
SEED = 0

import json, os
os.makedirs(OUT_DIR, exist_ok=True)
PROTOCOL = {k: v for k, v in dict(globals()).items() if k.isupper() and isinstance(v, (str, int, float, list, dict, tuple))}
with open(os.path.join(OUT_DIR, "protocol.json"), "w") as f:
    json.dump(PROTOCOL, f, indent=1)
print("protocol written before the run:", os.path.join(OUT_DIR, "protocol.json"))

In [ ]:
# Google Drive: results are copied to MyDrive/genco_runs at the end (Colab asks for permission once)
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/genco_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("results will be saved to", DRIVE_DIR)
except Exception as e:  # not running on Colab, or Drive declined
    DRIVE_DIR = None
    print("Google Drive not available, results stay in the session:", e)


def save_to_drive(path):
    """Copies a result file to Google Drive and starts a browser download as well."""
    if DRIVE_DIR:
        shutil.copy(path, DRIVE_DIR)
        print(f"saved {os.path.basename(path)} to {DRIVE_DIR}")
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        pass

In [ ]:
# Install and import: gridfm-graphkit at the pinned commit, lightsim2grid, and the verification module of this repository
import importlib.util, subprocess, sys, time, base64


def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow",
        "networkx": "networkx", "yaml": "pyyaml", "lightsim2grid": "lightsim2grid"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric
if importlib.util.find_spec("torch_scatter") is None:   # a small pure-PyTorch stand-in for the functions graphkit uses
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('''"""Pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""
import torch

__version__ = "0.0.0-shim"


def _prep(src, index, dim):
    dim = dim if dim >= 0 else src.dim() + dim
    if index.dim() != src.dim():
        shape = [1] * src.dim()
        shape[dim] = -1
        index = index.view(shape).expand_as(src)
    return index, dim


def _size(src, index, dim, dim_size):
    size = list(src.shape)
    if dim_size is None:
        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0
    size[dim] = dim_size
    return size


def scatter_add(src, index, dim=-1, out=None, dim_size=None):
    index, dim = _prep(src, index, dim)
    if out is None:
        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)
    return out.scatter_add_(dim, index, src)


scatter_sum = scatter_add


def scatter_mean(src, index, dim=-1, out=None, dim_size=None):
    total = scatter_add(src, index, dim, None, dim_size)
    idx, d = _prep(src, index, dim)
    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)
    return total / count


def scatter_max(src, index, dim=-1, out=None, dim_size=None):
    idx, d = _prep(src, index, dim)
    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)
    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)
    res[torch.isinf(res)] = 0
    return res, None


def scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):
    if reduce in ("sum", "add"):
        return scatter_add(src, index, dim, out, dim_size)
    if reduce == "mean":
        return scatter_mean(src, index, dim, out, dim_size)
    if reduce == "max":
        return scatter_max(src, index, dim, out, dim_size)[0]
    raise NotImplementedError(reduce)
''')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT, PD_H, QD_H
from torch_geometric.loader import DataLoader

GENCO_VERIFY_FILES = {'__init__.py': 'IiIiZ2VuY29fdmVyaWZ5OiB2ZXJpZmllZCBOLWsgY29udGluZ2VuY3kgc2NyZWVuaW5nIHdpdGggR0VOQ08gYXMgdGhlIHN0YXJ0aW5nIHBvaW50LiIiIgoKZnJvbSAuY2FzZXMgaW1wb3J0IGNhc2VfZnJvbV90YWJsZXMsIHN5bnRoZXRpY19jYXNlCmZyb20gLnBvd2VyZmxvdyBpbXBvcnQgSEFWRV9LTFUsIGJ1aWxkX3lidXMsIG5ld3Rvbl9zb2x2ZQpmcm9tIC52ZXJpZmllciBpbXBvcnQgREVGQVVMVF9QT0xJQ1ksIFNUQVRVU0VTLCBTdGFydEF0dGVtcHQsIFZlcmlmaWVyLCBWZXJpZnlSZXN1bHQKCl9fdmVyc2lvbl9fID0gIjEuMS4wIgpfX2FsbF9fID0gWwogICAgIlZlcmlmaWVyIiwKICAgICJWZXJpZnlSZXN1bHQiLAogICAgIlN0YXJ0QXR0ZW1wdCIsCiAgICAiREVGQVVMVF9QT0xJQ1kiLAogICAgIlNUQVRVU0VTIiwKICAgICJjYXNlX2Zyb21fdGFibGVzIiwKICAgICJzeW50aGV0aWNfY2FzZSIsCiAgICAibmV3dG9uX3NvbHZlIiwKICAgICJidWlsZF95YnVzIiwKICAgICJIQVZFX0tMVSIsCl0K', 'cases.py': 'IiIiQnVpbGRpbmcgY2FzZSBkaWN0aW9uYXJpZXM6IGZyb20gdGhlIHJlbGVhc2VkIEdFTkNPIHRhYmxlcywgb3IgYSB0aHJlZS1idXMgdGVzdCBncmlkLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5wb3dlcmZsb3cgaW1wb3J0IFNOX01WQQoKX0JVU19DT0xVTU5TID0gWyJQZCIsICJRZCIsICJQZyIsICJRZyIsICJWbSIsICJWYSIsICJtaW5fdm1fcHUiLCAibWF4X3ZtX3B1IiwgIkdTIiwgIkJTIl0KCgpkZWYgY2FzZV9mcm9tX3RhYmxlcyhidXMsIGJyYW5jaCwgcmVtb3ZlZF9pZHg9KCksIGdlbj1Ob25lKToKICAgICIiIkEgY2FzZSBkaWN0aW9uYXJ5IGZyb20gb25lIHNjZW5hcmlvIG9mIHRoZSByZWxlYXNlZCBidXMgYW5kIGJyYW5jaCB0YWJsZXMgKHBhbmRhcyBEYXRhRnJhbWVzKS4KCiAgICBgcmVtb3ZlZF9pZHhgIGxpc3RzIGJyYW5jaCBpbmRpY2VzIHRvIHRha2Ugb3V0IG9mIHNlcnZpY2UgKHRoZSBjb250aW5nZW5jeSkuIFdpdGggdGhlIGdlbmVyYXRvciB0YWJsZSwgcGVyLWJ1cwogICAgcmVhY3RpdmUgbGltaXRzIGFyZSBhZGRlZCBzbyB0aGF0IGBlbmZvcmNlX3FfbGltaXRzYCBjYW4gYmUgdXNlZC4KICAgICIiIgogICAgYnVzID0gYnVzLnNvcnRfdmFsdWVzKCJidXMiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBicmFuY2ggPSBicmFuY2guc29ydF92YWx1ZXMoImlkeCIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIG5iID0gbGVuKGJ1cykKICAgIGQgPSB7YzogYnVzW2NdLnRvX251bXB5KG5wLmZsb2F0NjQpIGZvciBjIGluIF9CVVNfQ09MVU1OU30KICAgIGZvciBjIGluICgiUFEiLCAiUFYiLCAiUkVGIik6CiAgICAgICAgZFtjXSA9IGJ1c1tjXS50b19udW1weShucC5mbG9hdDY0KSA+IDAuNQogICAgaWYgIlZhX2RjIiBpbiBidXM6CiAgICAgICAgZFsiVmFfZGMiXSA9IGJ1c1siVmFfZGMiXS50b19udW1weShucC5mbG9hdDY0KQogICAgZFsibmIiXSA9IG5iCiAgICBkWyJmIl0gPSBicmFuY2hbImZyb21fYnVzIl0udG9fbnVtcHkoKS5hc3R5cGUobnAuaW50NjQpCiAgICBkWyJ0Il0gPSBicmFuY2hbInRvX2J1cyJdLnRvX251bXB5KCkuYXN0eXBlKG5wLmludDY0KQogICAgZm9yIHkgaW4gKCJZZmYiLCAiWWZ0IiwgIll0ZiIsICJZdHQiKToKICAgICAgICBkW3ldID0gYnJhbmNoW3kgKyAiX3IiXS50b19udW1weShucC5mbG9hdDY0KSArIDFqICogYnJhbmNoW3kgKyAiX2kiXS50b19udW1weShucC5mbG9hdDY0KQogICAgZFsicmF0ZSJdID0gYnJhbmNoWyJyYXRlX2EiXS50b19udW1weShucC5mbG9hdDY0KQogICAgZFsibGluZSJdID0gYnJhbmNoWyJpZHgiXS50b19udW1weSgpLmFzdHlwZShucC5pbnQ2NCkKICAgIGRbIm9uIl0gPSBicmFuY2hbImJyX3N0YXR1cyJdLnRvX251bXB5KG5wLmZsb2F0NjQpID4gMC41CiAgICBpZiBsZW4ocmVtb3ZlZF9pZHgpOgogICAgICAgIGRbIm9uIl0gJj0gfm5wLmlzaW4oZFsibGluZSJdLCBucC5hc2FycmF5KHJlbW92ZWRfaWR4LCBkdHlwZT1ucC5pbnQ2NCkpCiAgICAgICAgZC5wb3AoIlZhX2RjIiwgTm9uZSkgICMgc3RhbGUgZm9yIHRoZSBuZXcgdG9wb2xvZ3k7IHRoZSBWZXJpZmllciByZWNvbXB1dGVzIGl0CiAgICBpZiBnZW4gaXMgbm90IE5vbmU6CiAgICAgICAgZyA9IGdlbltnZW5bImluX3NlcnZpY2UiXS50b19udW1weShucC5mbG9hdDY0KSA+IDAuNV0KICAgICAgICBidXNlcyA9IGdbImJ1cyJdLnRvX251bXB5KCkuYXN0eXBlKGludCkKICAgICAgICBkWyJxbWluX3B1Il0sIGRbInFtYXhfcHUiXSA9IG5wLnplcm9zKG5iKSwgbnAuemVyb3MobmIpCiAgICAgICAgbnAuYWRkLmF0KGRbInFtaW5fcHUiXSwgYnVzZXMsIGdbIm1pbl9xX212YXIiXS50b19udW1weShucC5mbG9hdDY0KSAvIFNOX01WQSkKICAgICAgICBucC5hZGQuYXQoZFsicW1heF9wdSJdLCBidXNlcywgZ1sibWF4X3FfbXZhciJdLnRvX251bXB5KG5wLmZsb2F0NjQpIC8gU05fTVZBKQogICAgcmV0dXJuIGQKCgpkZWYgc3ludGhldGljX2Nhc2UobG9hZF9tdz04MC4wLCBsb2FkX212YXI9MjAuMCwgcmF0ZV9tdmE9MTIwLjAsIHhfcHU9MC4xKToKICAgICIiIlRocmVlIGJ1c2VzOiBzbGFjayAoMS4wMiBwLnUuKSwgUFYgKDEuMCBwLnUuLCA0MCBNVywgYXQgbW9zdCA1IE12YXIpIGFuZCBhIFBRIGxvYWQ7IGEgdHJpYW5nbGUgb2YgbGluZXMuIiIiCiAgICBuYiA9IDMKICAgIHlzID0gMSAvICgwLjAxICsgMWogKiB4X3B1KQogICAgcmV0dXJuIHsKICAgICAgICAibmIiOiBuYiwKICAgICAgICAiZiI6IG5wLmFycmF5KFswLCAxLCAwXSksCiAgICAgICAgInQiOiBucC5hcnJheShbMSwgMiwgMl0pLAogICAgICAgICJZZmYiOiBucC5mdWxsKDMsIHlzKSwKICAgICAgICAiWWZ0IjogbnAuZnVsbCgzLCAteXMpLAogICAgICAgICJZdGYiOiBucC5mdWxsKDMsIC15cyksCiAgICAgICAgIll0dCI6IG5wLmZ1bGwoMywgeXMpLAogICAgICAgICJvbiI6IG5wLm9uZXMoMywgYm9vbCksCiAgICAgICAgInJhdGUiOiBucC5mdWxsKDMsIHJhdGVfbXZhKSwKICAgICAgICAibGluZSI6IG5wLmFyYW5nZSgzKSwKICAgICAgICAiR1MiOiBucC56ZXJvcyhuYiksCiAgICAgICAgIkJTIjogbnAuemVyb3MobmIpLAogICAgICAgICJQZyI6IG5wLmFycmF5KFswLjAsIDQwLjAsIDAuMF0pLAogICAgICAgICJQZCI6IG5wLmFycmF5KFswLjAsIDAuMCwgbG9hZF9td10pLAogICAgICAgICJRZyI6IG5wLnplcm9zKG5iKSwKICAgICAgICAiUWQiOiBucC5hcnJheShbMC4wLCAwLjAsIGxvYWRfbXZhcl0pLAogICAgICAgICJWbSI6IG5wLmFycmF5KFsxLjAyLCAxLjAsIDEuMF0pLAogICAgICAgICJWYSI6IG5wLnplcm9zKG5iKSwKICAgICAgICAiUFEiOiBucC5hcnJheShbRmFsc2UsIEZhbHNlLCBUcnVlXSksCiAgICAgICAgIlBWIjogbnAuYXJyYXkoW0ZhbHNlLCBUcnVlLCBGYWxzZV0pLAogICAgICAgICJSRUYiOiBucC5hcnJheShbVHJ1ZSwgRmFsc2UsIEZhbHNlXSksCiAgICAgICAgIm1pbl92bV9wdSI6IG5wLmZ1bGwobmIsIDAuOSksCiAgICAgICAgIm1heF92bV9wdSI6IG5wLmZ1bGwobmIsIDEuMSksCiAgICAgICAgInFtaW5fcHUiOiBucC5hcnJheShbLTEwLjAsIC0wLjUsIDAuMF0pLAogICAgICAgICJxbWF4X3B1IjogbnAuYXJyYXkoWzEwLjAsIDAuMDUsIDAuMF0pLAogICAgfQo=', 'powerflow.py': 'IiIiQUMgcG93ZXItZmxvdyBwcmltaXRpdmVzOiBhZG1pdHRhbmNlIG1hdHJpeCwgcG93ZXIgbWlzbWF0Y2ggYW5kIE5ld3Rvbi1SYXBoc29uLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBzY2lweS5zcGFyc2UgYXMgc3AKZnJvbSBzY2lweS5zcGFyc2UubGluYWxnIGltcG9ydCBzcGx1Cgp0cnk6CiAgICBmcm9tIGxpZ2h0c2ltMmdyaWQuYWxnb3JpdGhtIGltcG9ydCBOUl9LTFUgYXMgX0tMVQpleGNlcHQgSW1wb3J0RXJyb3I6ICAjIHByYWdtYTogbm8gY292ZXIgLSBvbGRlciBsaWdodHNpbTJncmlkIHJlbGVhc2VzCiAgICB0cnk6CiAgICAgICAgZnJvbSBsaWdodHNpbTJncmlkLnNvbHZlciBpbXBvcnQgS0xVU29sdmVyIGFzIF9LTFUKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBfS0xVID0gTm9uZQoKSEFWRV9LTFUgPSBfS0xVIGlzIG5vdCBOb25lClNOX01WQSA9IDEwMC4wCgoKZGVmIGJ1aWxkX3lidXMobmIsIGYsIHQsIFlmZiwgWWZ0LCBZdGYsIFl0dCwgR1MsIEJTKToKICAgICIiIkJ1cyBhZG1pdHRhbmNlIG1hdHJpeCBmcm9tIHBlci1icmFuY2ggYWRtaXR0YW5jZXMgYW5kIGJ1cyBzaHVudHMgKHAudS4pLiIiIgogICAgcm93cyA9IG5wLnJfW2YsIGYsIHQsIHQsIG5wLmFyYW5nZShuYildCiAgICBjb2xzID0gbnAucl9bZiwgdCwgZiwgdCwgbnAuYXJhbmdlKG5iKV0KICAgIHZhbHMgPSBucC5yX1tZZmYsIFlmdCwgWXRmLCBZdHQsIEdTICsgMWogKiBCU10KICAgIHJldHVybiBzcC5jc3JfbWF0cml4KCh2YWxzLCAocm93cywgY29scykpLCBzaGFwZT0obmIsIG5iKSkKCgpkZWYgbWF4X21pc21hdGNoKFksIFMsIFYsIG5vbnJlZiwgcHEpOgogICAgIiIiTGFyZ2VzdCBwb3dlciBtaXNtYXRjaDogUCBhdCBldmVyeSBub24tc2xhY2sgYnVzLCBRIGF0IHRoZSBQUSBidXNlcyAocC51LikuIiIiCiAgICBtID0gViAqIG5wLmNvbmooWSBAIFYpIC0gUwogICAgcCA9IG5wLmFicyhtLnJlYWxbbm9ucmVmXSkubWF4KCkgaWYgbGVuKG5vbnJlZikgZWxzZSAwLjAKICAgIHEgPSBucC5hYnMobS5pbWFnW3BxXSkubWF4KCkgaWYgbGVuKHBxKSBlbHNlIDAuMAogICAgcmV0dXJuIGZsb2F0KG1heChwLCBxKSkKCgpkZWYgX25ld3Rvbl9zY2lweShZLCBTLCBWMCwgcHYsIHBxLCB0b2wsIG1heF9pdCk6CiAgICAiIiJQb2xhciBOZXd0b24tUmFwaHNvbiB3aXRoIGEgc3BhcnNlIExVIGZhY3RvcmlzYXRpb24uIFJldHVybnMgKFYsIGNvbnZlcmdlZCwgaXRlcmF0aW9ucykuIiIiCiAgICBWID0gbnAuYXNhcnJheShWMCwgY29tcGxleCkuY29weSgpCiAgICBwdnBxID0gbnAucl9bcHYsIHBxXQogICAgbl9hbmcgPSBsZW4ocHZwcSkKICAgIFljID0gWS50b2NzYygpCiAgICBmb3IgaXQgaW4gcmFuZ2UobWF4X2l0ICsgMSk6CiAgICAgICAgbSA9IFYgKiBucC5jb25qKFljIEAgVikgLSBTCiAgICAgICAgRiA9IG5wLnJfW20ucmVhbFtwdnBxXSwgbS5pbWFnW3BxXV0KICAgICAgICBpZiBsZW4oRikgPT0gMCBvciBucC5tYXgobnAuYWJzKEYpKSA8IHRvbDoKICAgICAgICAgICAgcmV0dXJuIFYsIFRydWUsIGl0CiAgICAgICAgaWYgaXQgPT0gbWF4X2l0OgogICAgICAgICAgICBicmVhawogICAgICAgIGliID0gWWMgQCBWCiAgICAgICAgZFYgPSBzcC5kaWFncyhWKQogICAgICAgIGRTX2RWYSA9IDFqICogZFYgQCBucC5jb25qKHNwLmRpYWdzKGliKSAtIFljIEAgZFYpCiAgICAgICAgZFNfZFZtID0gZFYgQCBucC5jb25qKFljIEAgc3AuZGlhZ3MoViAvIG5wLmFicyhWKSkpICsgbnAuY29uaihzcC5kaWFncyhpYikpIEAgc3AuZGlhZ3MoViAvIG5wLmFicyhWKSkKICAgICAgICBKID0gc3AudnN0YWNrKAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgICBzcC5oc3RhY2soW2RTX2RWYVtwdnBxXVs6LCBwdnBxXS5yZWFsLCBkU19kVm1bcHZwcV1bOiwgcHFdLnJlYWxdKSwKICAgICAgICAgICAgICAgIHNwLmhzdGFjayhbZFNfZFZhW3BxXVs6LCBwdnBxXS5pbWFnLCBkU19kVm1bcHFdWzosIHBxXS5pbWFnXSksCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIGZvcm1hdD0iY3NjIiwKICAgICAgICApCiAgICAgICAgdHJ5OgogICAgICAgICAgICBkeCA9IHNwbHUoSikuc29sdmUoLUYpCiAgICAgICAgZXhjZXB0IChSdW50aW1lRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICByZXR1cm4gViwgRmFsc2UsIGl0CiAgICAgICAgdmEsIHZtID0gbnAuYW5nbGUoViksIG5wLmFicyhWKQogICAgICAgIHZhW3B2cHFdICs9IGR4WzpuX2FuZ10KICAgICAgICB2bVtwcV0gKz0gZHhbbl9hbmc6XQogICAgICAgIFYgPSB2bSAqIG5wLmV4cCgxaiAqIHZhKQogICAgICAgIGlmIG5vdCBucC5hbGwobnAuaXNmaW5pdGUoVikpOgogICAgICAgICAgICByZXR1cm4gViwgRmFsc2UsIGl0CiAgICByZXR1cm4gViwgRmFsc2UsIG1heF9pdAoKCmRlZiBuZXd0b25fc29sdmUoWSwgUywgVjAsIHJlZiwgcHYsIHBxLCB0b2wsIG1heF9pdCk6CiAgICAiIiJOZXd0b24tUmFwaHNvbiBmcm9tIFYwLCB1c2luZyBsaWdodHNpbTJncmlkJ3MgS0xVIHNvbHZlciB3aGVuIGluc3RhbGxlZC4KCiAgICBSZXR1cm5zIChWLCBjb252ZXJnZWQsIGl0ZXJhdGlvbnMpLgogICAgIiIiCiAgICBpZiBub3QgSEFWRV9LTFU6CiAgICAgICAgcmV0dXJuIF9uZXd0b25fc2NpcHkoWSwgUywgVjAsIHB2LCBwcSwgdG9sLCBtYXhfaXQpCiAgICB3ZWlnaHRzID0gbnAuemVyb3MoWS5zaGFwZVswXSkKICAgIHdlaWdodHNbcmVmXSA9IDEuMCAvIGxlbihyZWYpCiAgICBzb2x2ZXIgPSBfS0xVKCkKICAgIG9rID0gc29sdmVyLnNvbHZlKAogICAgICAgIFkudG9jc2MoKSwKICAgICAgICBucC5hc2FycmF5KFYwLCBjb21wbGV4KSwKICAgICAgICBucC5hc2FycmF5KFMsIGNvbXBsZXgpLAogICAgICAgIG5wLmFzYXJyYXkocmVmLCBucC5pbnQzMiksCiAgICAgICAgd2VpZ2h0cywKICAgICAgICBucC5hc2FycmF5KHB2LCBucC5pbnQzMiksCiAgICAgICAgbnAuYXNhcnJheShwcSwgbnAuaW50MzIpLAogICAgICAgIGludChtYXhfaXQpLAogICAgICAgIGZsb2F0KHRvbCksCiAgICApCiAgICBWID0gbnAuYXJyYXkoc29sdmVyLmdldF9WKCksIGR0eXBlPWNvbXBsZXgsIGNvcHk9VHJ1ZSkKICAgIHJldHVybiBWLCBib29sKG9rKSwgaW50KHNvbHZlci5nZXRfbmJfaXRlcigpKQo=', 'verifier.py': '...'}
pkg = os.path.abspath(os.path.join("modules", "genco_verify"))
os.makedirs(pkg, exist_ok=True)
for name, blob in GENCO_VERIFY_FILES.items():
    with open(os.path.join(pkg, name), "wb") as f:
        f.write(base64.b64decode(blob))
sys.path.insert(0, os.path.dirname(pkg))
import genco_verify
from genco_verify import Verifier, case_from_tables
from genco_verify.powerflow import HAVE_KLU, SN_MVA, build_ybus, newton_solve

import numpy as np, pandas as pd, pyarrow as pa, pyarrow.dataset as ds, scipy.sparse as sp, networkx as nx
from scipy.sparse.csgraph import connected_components, dijkstra
from pathlib import Path
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | torch_geometric {torch_geometric.__version__} | graphkit {GRAPHKIT_COMMIT[:7]} | "
      f"genco_verify {genco_verify.__version__} (lightsim2grid KLU: {HAVE_KLU}) | device {DEVICE}")

## 1. Downloads: the model, the authors' split and test metrics, and a spread of data partitions

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
SPLITS_FILE = f"{EVAL_RUN}/artifacts/stats/{NETWORK}_scenario_splits.json"
RMSE_FILE = f"{EVAL_RUN}/artifacts/test/{NETWORK}_RMSE.csv"
for p in [f"{MODEL_DIR}/best_model_state_dict.pt", f"{MODEL_DIR}/normalizer_stats.pt", f"{MODEL_DIR}/{MODEL_CONFIG}", SPLITS_FILE, RMSE_FILE]:
    if not (Path(LOCAL_MODEL) / p).exists():
        hf_hub_download(MODEL_REPO, p, revision=MODEL_REVISION, local_dir=LOCAL_MODEL)
TABLES = ["bus_data", "gen_data", "branch_data"]
ALL_PARTS = sorted(int(e.path.rsplit("=", 1)[1]) for e in api.list_repo_tree(
    DATA_REPO, path_in_repo="bus_data.parquet", repo_type="dataset", revision=DATA_REVISION) if "scenario_partition=" in e.path)
PARTS = sorted({ALL_PARTS[i] for i in np.linspace(0, len(ALL_PARTS) - 1, N_PARTITIONS).round().astype(int)})
t0 = time.perf_counter()
for table in TABLES:
    for p in PARTS:
        folder = f"{table}.parquet/scenario_partition={p}"
        for e in api.list_repo_tree(DATA_REPO, path_in_repo=folder, repo_type="dataset", revision=DATA_REVISION):
            if e.path.endswith(".parquet") and not (Path(LOCAL_DATA) / e.path).exists():
                hf_hub_download(DATA_REPO, e.path, repo_type="dataset", revision=DATA_REVISION, local_dir=LOCAL_DATA)
size_mb = sum(f.stat().st_size for f in Path(LOCAL_DATA).rglob("*.parquet")) / 1e6
print(f"{len(PARTS)} of {len(ALL_PARTS)} partitions ({PARTS[0]} to {PARTS[-1]}), {size_mb:.0f} MB, {time.perf_counter() - t0:.0f} s")

## 2. Operating points from the authors' test split

In [ ]:
def table(name, scen=None, columns=None):
    dset = ds.dataset(str(Path(LOCAL_DATA) / f"{name}.parquet"), format="parquet", partitioning="hive")
    flt = None
    if scen is not None:
        flt = ds.field("scenario").isin(pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type))
    t = dset.to_table(columns=columns, filter=flt).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


splits = json.load(open(Path(LOCAL_MODEL) / SPLITS_FILE))
test_key = next(k for k in splits if "test" in k.lower())
TEST = {int(s) for s in splits[test_key]}
OTHER = {int(s) for k, v in splits.items() if k != test_key for s in v}
assert not TEST & OTHER, "the test split overlaps the training or validation split"
br = table("branch_data", columns=["scenario", "br_status"])
on_disk = np.unique(br.scenario.to_numpy().astype(np.int64))
lines_out = br.groupby("scenario").br_status.apply(lambda s: int((s < 0.5).sum()))
pool = np.array(sorted(s for s in on_disk if s in TEST and lines_out[s] == 0), dtype=np.int64)
rng = np.random.default_rng(SEED)
BASE_IDS = np.sort(rng.choice(pool, size=min(N_BASES, len(pool)), replace=False))
SELECTION = {"splits": {k: len(v) for k, v in splits.items()}, "scenarios_on_disk": int(len(on_disk)),
             "on_disk_in_test_split": int(sum(s in TEST for s in on_disk)), "test_with_every_line_in_service": int(len(pool)),
             "base_points": int(len(BASE_IDS))}
print(SELECTION)

BUS_T = table("bus_data", BASE_IDS)
GEN_T = table("gen_data", BASE_IDS)
BR_T = table("branch_data", BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")
BASE = {}
for b in BASE_IDS:
    bus = BG.get_group(b).sort_values("bus").reset_index(drop=True)
    BASE[int(b)] = case_from_tables(bus, RG.get_group(b), gen=GG.get_group(b))
NB = next(iter(BASE.values()))["nb"]
SELECTION["generators_out_of_service"] = int((GEN_T.in_service < 0.5).sum())
print(f"{len(BASE)} operating points, {NB} buses, {len(next(iter(BASE.values()))['f'])} branches; "
      f"{SELECTION['generators_out_of_service']} generator outages among them (the dataset's N-1 variants)")

## 3. The model, and two checks before any contingency is built

In [ ]:
import yaml
with open(Path(LOCAL_MODEL) / MODEL_DIR / MODEL_CONFIG) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(Path(LOCAL_MODEL) / MODEL_DIR / "normalizer_stats.pt", map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(Path(LOCAL_MODEL) / MODEL_DIR / "best_model_state_dict.pt", map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print(f"GENCO loaded: {sum(p.numel() for p in MODEL.parameters()):,} parameters | normalizer baseMVA {NORMALIZER.baseMVA:.2f}")


def build_graph(sid, base, removed_lines=(), load_scale=1.0):
    """GENCO's input graph for operating point `base` with `removed_lines` (branch idx) out of service; loads scaled."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    g = TRANSFORM(g)
    if load_scale != 1.0:
        g["bus"].x = g["bus"].x.clone()
        g["bus"].x[:, PD_H] *= load_scale
        g["bus"].x[:, QD_H] *= load_scale
    return g


def genco(graphs):
    """GENCO's (Vm, Va in degrees) for each graph, and the inference time per graph (batch 1)."""
    out = []
    for g in graphs:
        g["scenario_id"] = torch.tensor([0], dtype=torch.long)
        b = next(iter(DataLoader([g], batch_size=1))).to(DEVICE)
        t0 = time.perf_counter()
        with torch.no_grad():
            o = MODEL(b.x_dict, b.edge_index_dict, b.edge_attr_dict, b.mask_dict)["bus"].float()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        ms = 1000 * (time.perf_counter() - t0)
        out.append((o[:, VM_OUT].cpu().numpy().astype(np.float64), np.degrees(o[:, VA_OUT].cpu().numpy().astype(np.float64)), ms))
    return out


# check 1: our network model reproduces the stored AC solutions (largest power mismatch of each stored state)
net_mis = np.array([Verifier(d, POLICY).mismatch(Verifier(d, POLICY).starts["base"]) for d in BASE.values()])
# check 2: GENCO reproduces the authors' test accuracy on these operating points
outs = genco([build_graph(b, b) for b in BASE_IDS])
err_pq = np.concatenate([(vm - BASE[int(b)]["Vm"])[BASE[int(b)]["PQ"]] for b, (vm, va, ms) in zip(BASE_IDS, outs)])
RMSE_VM_PQ = float(np.sqrt(np.mean(err_pq ** 2)))
authors = pd.read_csv(Path(LOCAL_MODEL) / RMSE_FILE).set_index("Metric")
MODEL_CHECK = {"network_max_mismatch_pu": float(net_mis.max()), "network_check_pu": NETWORK_CHECK_PU,
               "genco_rmse_vm_pq": RMSE_VM_PQ, "authors_rmse_vm_pq": float(authors.loc["RMSE-PQ", "Vm (p.u.)"]),
               "base_points": int(len(BASE_IDS))}
print(MODEL_CHECK)
if MODEL_CHECK["network_max_mismatch_pu"] > NETWORK_CHECK_PU:
    raise RuntimeError("Our network model does not reproduce the stored AC solutions; check the downloaded data partitions.")
if RMSE_VM_PQ > MODEL_CHECK_FACTOR * MODEL_CHECK["authors_rmse_vm_pq"]:
    raise RuntimeError("GENCO is far less accurate than the authors report: the model did not load correctly; check the installation and the downloaded checkpoint.")
print("both checks passed")

## 4. Contingencies (frozen construction)

In [ ]:
def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


d0 = BASE[int(BASE_IDS[0])]
G = nx.Graph()
G.add_edges_from(zip(d0["f"][d0["on"]].tolist(), d0["t"][d0["on"]].tolist()))
EB = nx.edge_betweenness_centrality(G, normalized=True)
LINE_BTW = np.array([EB.get((int(f_), int(t_)), EB.get((int(t_), int(f_)), 0.0)) for f_, t_ in zip(d0["f"], d0["t"])])


def pick_lines(d, k, mode, rng):
    """Rows of `k` lines to take out, keeping the grid in one piece (as on Texas)."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    centre = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=centre)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    if mode == "nearest":
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    elif mode == "uniform":
        order = rng.permutation(cand)
    elif mode == "corridor":
        h = REGION_HOPS
        while (dist <= h).sum() < 3 * k and h < 12:
            h += 1
        reg = cand[dist <= h]
        order = reg[np.lexsort((rng.random(len(reg)), -LINE_BTW[reg]))]
    else:
        raise ValueError(mode)
    removed = []
    for i in order:
        if len(removed) == k:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < k:
        return None, np.nan, centre
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, centre


CASES, SKIPPED = [], 0
for b in BASE_IDS:
    d = BASE[int(b)]
    rng = np.random.default_rng(SEED * 1000003 + int(b))
    for j, (mode, k) in enumerate(FAMILIES):
        rem, radius, centre = pick_lines(d, k, mode, rng)
        if rem is None:
            SKIPPED += 1; continue
        d2 = {kk: (v.copy() if isinstance(v, np.ndarray) else v) for kk, v in d.items()}
        d2["on"][rem] = False
        d2.pop("Va_dc", None)
        CASES.append({"sid": int(b) * 10 + j, "base": int(b), "family": f"{mode}-N{k}", "mode": mode, "k": k,
                      "removed_lines": d["line"][rem].astype(int).tolist(), "radius_hops": radius, "centre": centre, "d": d2})
print(f"{len(CASES)} contingencies on {len(BASE_IDS)} operating points ({SKIPPED} could not be built without islanding)")

## 5. GENCO, the classical solver and the verified workflow on every case

In [ ]:
def decisions(limits):
    return {"over": set(limits["overloaded_lines"]), "low": set(limits["low_voltage_buses"]), "high": set(limits["high_voltage_buses"])}


def count(ref, got):
    """Decisions of the reference that `got` misses, and decisions in `got` that the reference does not have."""
    missed = {k: len(ref[k] - got[k]) for k in ref}
    extra = {k: len(got[k] - ref[k]) for k in ref}
    return missed, extra


rows = []
t0 = time.perf_counter()
for c in CASES:
    vm, va, ms_inf = genco([build_graph(c["sid"], c["base"], c["removed_lines"])])[0]
    ver = Verifier(c["d"], POLICY)
    rc = ver.run(policy_name="classical")
    rg = ver.run(genco_vm=vm, genco_va_deg=va, policy_name="genco_verify")
    g = rg.genco
    ref_run = rc if rc.status != "unresolved" else (rg if rg.status != "unresolved" else None)
    row = {k: c[k] for k in ("sid", "base", "family", "mode", "k", "radius_hops", "centre")}
    row.update({"removed_lines": json.dumps(c["removed_lines"]),
                "classical_status": rc.status, "classical_iterations": rc.iterations, "classical_ms": rc.timing_ms["total_ms"],
                "verified_status": rg.status, "verified_start": rg.start_used, "verified_iterations": rg.iterations,
                "verified_ms": rg.timing_ms["total_ms"] + ms_inf, "genco_inference_ms": ms_inf,
                "reference": None if ref_run is None else ("classical" if ref_run is rc else "verified"),
                "genco_min_vm_pq": float(np.min(np.asarray(g["vm"])[c["d"]["PQ"]])), "genco_n_over": g["n_over"],
                "genco_n_low_v": g["n_low_v"], "genco_n_high_v": g["n_high_v"], "genco_resP_p99_MW": g["resP_p99_MW"]})
    row.update({k: np.nan for k in ("ref_overloads", "ref_vviol", "genco_missed_overloads", "genco_missed_vviol", "genco_false_overloads",
                                    "genco_false_vviol", "genco_worst_err", "ref_min_vm_pq", "verified_missed_decisions")})
    if ref_run is not None:
        ref = decisions(ref_run.limits)
        missed, extra = count(ref, {"over": set(g["overloaded_lines"]), "low": set(g["low_voltage_buses"]),
                                    "high": set(g["high_voltage_buses"])})
        ref_vm = np.asarray(ref_run.state["vm"])
        row.update({"ref_overloads": len(ref["over"]), "ref_vviol": len(ref["low"]) + len(ref["high"]),
                    "genco_missed_overloads": missed["over"], "genco_missed_vviol": missed["low"] + missed["high"],
                    "genco_false_overloads": extra["over"], "genco_false_vviol": extra["low"] + extra["high"],
                    "genco_worst_err": float(np.max(np.abs(np.asarray(g["vm"]) - ref_vm)[c["d"]["PQ"]])),
                    "ref_min_vm_pq": float(np.min(ref_vm[c["d"]["PQ"]]))})
        if rg.status != "unresolved":
            vmiss, _ = count(ref, decisions(rg.limits))
            row["verified_missed_decisions"] = sum(vmiss.values())
    rows.append(row)
R = pd.DataFrame(rows)
R["resolved"] = R.reference.notna()
any_miss = (R.genco_missed_overloads > 0) | (R.genco_missed_vviol > 0)
miss_or_false = any_miss | (R.genco_false_overloads > 0) | (R.genco_false_vviol > 0)
R["genco_any_miss"] = np.where(R.resolved, any_miss.astype(float), np.nan)   # 1/0 on resolved cases, empty otherwise (as on Texas)
R["genco_miss_or_false_flag"] = np.where(R.resolved, miss_or_false.astype(float), np.nan)
R["false_alarm"] = R.resolved & (R.verified_status == "unresolved")
R["missed_alarm"] = (~R.resolved) & (R.verified_status != "unresolved")
print(f"{len(R)} cases in {time.perf_counter() - t0:.0f} s")

## 6. Results: the workflow, GENCO alone, and where it fails

In [ ]:
GROUP = {"uniform-N1": "one line", "uniform-N3": "random", "uniform-N6": "random",
         "nearest-N3": "clustered", "nearest-N6": "clustered", "corridor-N6": "clustered"}
R["group"] = R.family.map(GROUP)


def per(g):
    res = g[g.resolved]
    return pd.Series({
        "cases": len(g), "resolved": int(g.resolved.sum()), "unresolved": int((~g.resolved).sum()),
        "verified_decisions": int((g.verified_status != "unresolved").sum()), "false_alarms": int(g.false_alarm.sum()),
        "missed_alarms": int(g.missed_alarm.sum()),
        "verified_with_a_different_decision": int((res.verified_missed_decisions.fillna(0) > 0).sum()),
        "genco_alone_cases_with_a_miss": int(res.genco_any_miss.sum()), "genco_alone_cases_with_a_miss_or_false_flag": int(res.genco_miss_or_false_flag.sum()),
        "ref_overloads": int(res.ref_overloads.sum()), "genco_missed_overloads": int(res.genco_missed_overloads.sum()),
        "ref_vviol": int(res.ref_vviol.sum()), "genco_missed_vviol": int(res.genco_missed_vviol.sum()),
        "genco_worst_err_median": float(res.genco_worst_err.median()),
        "iterations_classical_mean": float(res.classical_iterations[res.classical_status != "unresolved"].mean()),
        "iterations_from_genco_mean": float(res.verified_iterations[res.verified_start == "genco"].mean()),
        "genco_start_used": int((g.verified_start == "genco").sum()),
        "ms_classical_median": float(g.classical_ms.median()), "ms_verified_median": float(g.verified_ms.median())})


BY_FAMILY = R.groupby("family", sort=False).apply(per, include_groups=False)
BY_GROUP = R.groupby("group", sort=False).apply(per, include_groups=False)
TOTAL = per(R)
pd.set_option("display.width", 200)
print(BY_FAMILY.T.to_string())
print(BY_GROUP.T.to_string())

# uncertainty: cluster bootstrap over operating points for the rates that carry the comparison
rng = np.random.default_rng(SEED + 11)
bases = R.base.unique()
by_base = {b: R[R.base == b] for b in bases}
boot = {grp: {"miss_rate": [], "err_median": []} for grp in GROUP.values()}
boot_ratio = []
for _ in range(BOOTSTRAP):
    s = pd.concat([by_base[b] for b in rng.choice(bases, len(bases))])
    s = s[s.resolved]
    for grp in set(GROUP.values()):
        x = s[s.group == grp]
        boot[grp]["miss_rate"].append(x.genco_any_miss.mean()); boot[grp]["err_median"].append(x.genco_worst_err.median())
    boot_ratio.append(s[s.group == "clustered"].genco_worst_err.median() / s[s.group == "random"].genco_worst_err.median())
CI = {grp: {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in d.items()} for grp, d in boot.items()}
res = R[R.resolved]
ratio = float(res[res.group == "clustered"].genco_worst_err.median() / res[res.group == "random"].genco_worst_err.median())
CI["clustered_over_random_err_ratio"] = [float(np.nanpercentile(boot_ratio, 2.5)), float(np.nanpercentile(boot_ratio, 97.5))]
un = R[~R.resolved]
UNRESOLVED = {"cases": int(len(un)), "genco_min_vm_pq_median_unresolved": float(un.genco_min_vm_pq.median()) if len(un) else None,
              "genco_min_vm_pq_median_resolved": float(res.genco_min_vm_pq.median()),
              "genco_predicts_no_violation": int(((un.genco_n_over == 0) & (un.genco_n_low_v == 0) & (un.genco_n_high_v == 0)).sum()),
              "verified_workflow_alarm": int((un.verified_status == "unresolved").sum())}
print(f"clustered / random median worst-bus error: x{ratio:.1f} (95 % CI x{CI['clustered_over_random_err_ratio'][0]:.1f} to x{CI['clustered_over_random_err_ratio'][1]:.1f})")
print("unresolved cases:", UNRESOLVED)

## 7. The load-growth probe

In [ ]:
def slope_through(x, y):
    """Least-squares slope of y on x (both centred), as on Texas."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


def ac_growth(d, growth):
    """|V| of the AC solution with every load scaled by (1 + growth); generator set-points fixed, the slack takes the rest."""
    on = d["on"]
    Y = build_ybus(d["nb"], d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA - growth * (d["Pd"] + 1j * d["Qd"]) / SN_MVA
    V0 = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    ref, pv, pq = (np.nonzero(d[k])[0] for k in ("REF", "PV", "PQ"))
    V, ok, _ = newton_solve(Y, S, V0, ref, pv, pq, 1e-10, 25)
    return np.abs(V) if ok else np.full(d["nb"], np.nan)


growth_rows = []
for b in BASE_IDS[:N_GROWTH]:
    d = BASE[int(b)]
    (vp, _, _), (vn, _, _) = genco([build_graph(b, b, load_scale=1 + GROWTH), build_graph(b, b, load_scale=1 - GROWTH)])
    g_sec = (vp - vn) / (2 * GROWTH)
    t_sec = (ac_growth(d, GROWTH) - ac_growth(d, -GROWTH)) / (2 * GROWTH)
    ok = d["PQ"] & np.isfinite(t_sec)
    growth_rows.append({"base": int(b), "slope_all_pq": slope_through(t_sec[ok], g_sec[ok]),
                        "corr_all_pq": float(np.corrcoef(t_sec[ok], g_sec[ok])[0, 1]),
                        "median_true_drop": float(np.median(t_sec[ok])), "median_genco_drop": float(np.median(g_sec[ok]))})
GROWTH_T = pd.DataFrame(growth_rows)
GROWTH_SUMMARY = {"operating_points": int(len(GROWTH_T)), "slope_median": float(GROWTH_T.slope_all_pq.median()),
                  "slope_iqr": [float(GROWTH_T.slope_all_pq.quantile(0.25)), float(GROWTH_T.slope_all_pq.quantile(0.75))],
                  "corr_median": float(GROWTH_T.corr_all_pq.median())}
print("GENCO's response to load growth relative to AC:", GROWTH_SUMMARY)

## 8. Output files

In [ ]:
import platform
R.to_csv(Path(OUT_DIR) / "cases.csv", index=False, float_format="%.6g")
BY_FAMILY.to_csv(Path(OUT_DIR) / "by_family.csv")
BY_GROUP.to_csv(Path(OUT_DIR) / "by_group.csv")
GROWTH_T.to_csv(Path(OUT_DIR) / "load_growth.csv", index=False, float_format="%.6g")
summary = {"protocol": PROTOCOL, "selection": SELECTION, "model_check": MODEL_CHECK,
           "cases": int(len(R)), "skipped_constructions": int(SKIPPED), "total": {k: (float(v) if isinstance(v, float) else int(v)) for k, v in TOTAL.items()},
           "clustered_over_random_err_ratio": ratio, "ci95": CI, "unresolved": UNRESOLVED, "load_growth": GROWTH_SUMMARY,
           "versions": {"python": platform.python_version(), "torch": torch.__version__, "torch_geometric": torch_geometric.__version__,
                        "numpy": np.__version__, "lightsim2grid": HAVE_KLU, "genco_verify": genco_verify.__version__,
                        "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}}
with open(Path(OUT_DIR) / "summary.json", "w") as f:
    json.dump(summary, f, indent=1)

t = TOTAL
print(f"- {len(R)} contingencies on {len(BASE_IDS)} test operating points of IEEE 118; the classical solver resolves {t.resolved:.0f}, {t.unresolved:.0f} unresolved")
print(f"- verified workflow: {t.verified_decisions:.0f} AC decisions, {t.false_alarms:.0f} false alarms, {t.missed_alarms:.0f} unresolved cases passed, "
      f"{t.verified_with_a_different_decision:.0f} verified cases with a decision different from the classical solution")
print(f"- GENCO alone: misses a decision in {t.genco_alone_cases_with_a_miss:.0f} of {t.resolved:.0f} resolved cases "
      f"(with false flags counted too: {t.genco_alone_cases_with_a_miss_or_false_flag:.0f})")
for grp, r_ in BY_GROUP.iterrows():
    print(f"  {grp:28s} median worst-bus error {r_.genco_worst_err_median:.4f} p.u.; cases with a missed decision {r_.genco_alone_cases_with_a_miss:.0f} of {r_.resolved:.0f}")
print(f"- unresolved cases where GENCO shows no violation at all: {UNRESOLVED['genco_predicts_no_violation']} of {UNRESOLVED['cases']}")
print(f"- load growth: GENCO responds {GROWTH_SUMMARY['slope_median']:.2f} times as strongly as AC (median over {GROWTH_SUMMARY['operating_points']} operating points)")
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"written {OUT_DIR}.zip")
save_to_drive(OUT_DIR + ".zip")